In [1]:
import yfinance as yf
import pandas as pd
import numpy as np

spy = yf.download(
    "BTC-USD",
    period = "60d",
    interval = "15m"
)

spy.head()

[*********************100%***********************]  1 of 1 completed


Price,Close,High,Low,Open,Volume
Ticker,BTC-USD,BTC-USD,BTC-USD,BTC-USD,BTC-USD
Datetime,,,,,
2026-08-11 00:00:00+00:00,63917.519531,63917.519531,63834.511719,63911.878906,100325376
2026-08-11 00:15:00+00:00,63921.839844,63933.238281,63851.718750,63914.871094,62926848
2026-08-11 00:30:00+00:00,63902.519531,63942.988281,63880.699219,63929.300781,129374208
2026-08-11 00:45:00+00:00,63971.199219,63989.980469,63916.949219,63916.949219,0
2026-08-11 01:00:00+00:00,63993.621094,64013.390625,63916.101562,63966.148438,220114944


In [2]:
spy.columns = [
    col[0] if col[1] == '' else col[0]
    for col in spy.columns
]
print(spy.columns)

Index(['Close', 'High', 'Low', 'Open', 'Volume'], dtype='str')


In [3]:
spy['log_returns'] = np.log(spy['Close']/spy['Close'].shift())
spy[['Close', 'log_returns']].head()

,Close,log_returns
Datetime,,
2026-08-11 00:00:00+00:00,63917.519531,NaN
2026-08-11 00:15:00+00:00,63921.839844,0.000068
2026-08-11 00:30:00+00:00,63902.519531,-0.000302
2026-08-11 00:45:00+00:00,63971.199219,0.001074
2026-08-11 01:00:00+00:00,63993.621094,0.000350


In [4]:
spy['lag_1'] = spy['log_returns'].shift()
spy['lag_2'] = spy['log_returns'].shift(2)
spy['lag_3'] = spy['log_returns'].shift(3)

spy = spy.dropna(subset = [
    'log_returns',
    'lag_1',
    'lag_2',
    'lag_3'
])

### Set up to train models

In [5]:
#Import torch stuff
import torch
import torch.nn as nn
import torch.optim as optim

In [16]:
#Train Model
def train_model(criterion, df_train, df_test, features, target, no_epochs=1000,verbose=False):

    no_features = len(features)
    model = nn.Linear(no_features, 1)
    optimizer = optim.SGD(model.parameters(),lr=0.01)

    X_train, X_test = df_train[features], df_test[features]
    y_train, y_test = df_train[target], df_test[target]

    X_train = torch.tensor(X_train.values, dtype=torch.float32)
    X_test = torch.tensor(X_test.values, dtype=torch.float32)

    y_train = torch.tensor(y_train.values, dtype=torch.float32).unsqueeze(1)
    y_test = torch.tensor(y_test.values, dtype=torch.float32).unsqueeze(1)
    
    for epoch in range(no_epochs):
        optimizer.zero_grad()
        y_pred = model(X_train)
        loss = criterion(y_pred,y_train)
        loss.backward()
        optimizer.step()

    return model

### First oder autoregressive model trained on Rolling window

In [35]:
# Features and tartgets
spy['abs_log_returns'] = spy['log_returns'].abs()
spy['abs_lag_1'] = spy['lag_1'].abs()
features = ['abs_lag_1']
target = 'abs_log_returns'

#Splitting test/ train data using rolling window of size 130
# Spliting data using time series split.
train_size = 0.6
i = int(len(spy) * train_size)
spy_train, spy_test = spy[:i].copy(), spy[i:].copy()

#Training the model
no_features = len(features)
model = nn.Linear(no_features, 1)

magnitude_model = train_model(nn.HuberLoss(), spy_train, spy_test,features,
            target, no_epochs=1000,verbose=False)

X_magnitude_test = torch.tensor(
    spy_test[features].values,
    dtype=torch.float32
)

with torch.no_grad():
    magnitude_prediction = magnitude_model(X_magnitude_test).flatten()

### Third Order Autoregresssive model train on Time Split Series

In [36]:
# Features and Targest
spy['log_return_dir'] = (spy['log_returns'] > 0).astype(int)
features = ['lag_1']#, 'lag_2', 'lag_3']
target = 'log_return_dir'

# Spliting data using time series split.
train_size = 0.6
i = int(len(spy) * train_size)
spy_train, spy_test = spy[:i].copy(), spy[i:].copy()

#EXPERIMENT START

feature_mean = spy_train[features].mean()
feature_std = spy_train[features].std()

spy_train_scaled = spy_train.copy()
spy_test_scaled = spy_test.copy()

spy_train_scaled[features] = (
    spy_train[features] - feature_mean
) / feature_std

spy_test_scaled[features] = (
    spy_test[features] - feature_mean
) / feature_std

#EXPERIMENT END

#Training the model
no_features = len(features)
model = nn.Linear(no_features, 1)

direction_model = train_model(nn.BCEWithLogitsLoss() ,spy_train_scaled, spy_test_scaled,features,
            target, no_epochs=1000,verbose=False)


X_direction_test = torch.tensor(
    spy_test_scaled[features].values,
    dtype=torch.float32
)

with torch.no_grad():
    direction_logits = direction_model(X_direction_test)
    direction_probability = torch.sigmoid(direction_logits).flatten()
direction = torch.where(
    direction_probability >= 0.5,
    1.0,
    -1.0
)

### Constructing the final model using direction and magnitude

In [37]:
print(len(spy_test))
print(len(direction_probability))
print(len(magnitude_prediction))

2293
2293
2293


In [38]:
# Predicted return
predicted_return =  direction * magnitude_prediction

spy_test['predicted_return'] = predicted_return.numpy()
spy_test[['log_returns', 'predicted_return']].head()

,log_returns,predicted_return
Datetime,,
2026-09-15 21:00:00+00:00,-0.000842,-0.000944
2026-09-15 21:15:00+00:00,-0.002421,0.000959
2026-09-15 21:30:00+00:00,0.000806,0.002092
2026-09-15 21:45:00+00:00,-0.003581,-0.000933
2026-09-15 22:00:00+00:00,-0.001363,0.002923


In [39]:
spy_test[['log_returns', 'predicted_return']].tail()

,log_returns,predicted_return
Datetime,,
2026-10-09 17:00:00+00:00,-0.001667,-0.000627
2026-10-09 17:15:00+00:00,0.000759,0.001551
2026-10-09 17:30:00+00:00,0.000232,-0.000899
2026-10-09 17:45:00+00:00,0.000242,-0.000521
2026-10-09 18:00:00+00:00,-0.000791,-0.000529


In [40]:
spy_test[['log_returns', 'predicted_return']].head(20)

,log_returns,predicted_return
Datetime,,
2026-09-15 21:00:00+00:00,-0.000842,-0.000944
2026-09-15 21:15:00+00:00,-0.002421,0.000959
2026-09-15 21:30:00+00:00,0.000806,0.002092
2026-09-15 21:45:00+00:00,-0.003581,-0.000933
2026-09-15 22:00:00+00:00,-0.001363,0.002923
2026-09-15 22:15:00+00:00,0.001625,0.001332
2026-09-15 22:30:00+00:00,0.001660,-0.001521
2026-09-15 22:45:00+00:00,0.000676,-0.001546
2026-09-15 23:00:00+00:00,0.003047,-0.000840


In [41]:
print("Direction probability statistics:")
print("Minimum:", direction_probability.min().item())
print("Maximum:", direction_probability.max().item())
print("Mean:", direction_probability.mean().item())
print("Std:", direction_probability.std().item())

print("\nFirst 20 probabilities:")
print(direction_probability[:20])

print("\nPredicted directions:")
print(torch.bincount((direction == 1).long()))

print("\nActual test directions:")
print(spy_test[target].value_counts())

Direction probability statistics:
Minimum: 0.18982401490211487
Maximum: 0.7255384922027588
Mean: 0.49463900923728943
Std: 0.041331350803375244

First 20 probabilities:
tensor([0.4764, 0.5149, 0.5513, 0.4768, 0.5776, 0.5269, 0.4579, 0.4571, 0.4798,
        0.4255, 0.5455, 0.5113, 0.5081, 0.4785, 0.4484, 0.5108, 0.5051, 0.4915,
        0.4678, 0.5613])

Predicted directions:
tensor([1295,  998])

Actual test directions:
log_return_dir
0    1149
1    1144
Name: count, dtype: int64


In [42]:
print("Weights:")
print(direction_model.weight.data)

print("\nBias:")
print(direction_model.bias.data)

Weights:
tensor([[-0.1885]])

Bias:
tensor([-0.0229])


In [43]:
# Compare the signs of actual and predicted returns
actual_sign = np.sign(spy_test['log_returns'])
predicted_sign = np.sign(spy_test['predicted_return'])

# Percentage of predictions where the signs are the same
directional_accuracy = (actual_sign == predicted_sign).mean()

print(f"Directional Accuracy: {directional_accuracy:.2%}")

Directional Accuracy: 50.20%
